# 09 promotion repurchase 2x2 EDA

Descriptive EDA only. No modeling, predictions, SHAP, Optuna, p-values, causal claims, plots, or feature engineering.


In [1]:
from pathlib import Path
from datetime import datetime
import math, subprocess, zipfile
import numpy as np
import pandas as pd

STEP='09_promotion_repurchase_2x2_eda_260513'
actual=subprocess.check_output(['git','rev-parse','--show-toplevel'], text=True).strip()
print('repo root:', actual)
if actual.replace('\\','/')!='C:/Code/ott-churn-prediction':
    raise SystemExit(f'STOP: repo root mismatch: {actual}')
repo=Path(actual).resolve()
park=repo/'park.ingyeom'
source=park/'data'/'(광일)Membership_v2_with_derived_features.csv'
nb_path=park/'notebook'/STEP/f'{STEP}.ipynb'
base_out=park/'reports'/'eda'/STEP
out=base_out/(datetime.now().strftime('run_%Y%m%d_%H%M%S')) if base_out.exists() and any(base_out.iterdir()) else base_out
out.mkdir(parents=True, exist_ok=True)
zip_dir=park/'zip'; zip_dir.mkdir(parents=True, exist_ok=True)
zip_path=zip_dir/f'{STEP}_review_package.zip'
print('output folder:', out)

def inside(p, parent):
    try:
        Path(p).resolve().relative_to(Path(parent).resolve()); return True
    except ValueError:
        return False
def rd(p): return pd.read_csv(p)
def first_col_set(df):
    for c in ['column_name','feature_name','name','column']:
        if c in df.columns: return set(df[c].dropna().astype(str))
    return set()

p06=park/'reports'/'audits'/'06_common_preprocessing_and_final_cohort_260513'
p05b=park/'reports'/'audits'/'05b_column_role_dictionary_patch_260513'
p07=park/'reports'/'audits'/'07_AARRR_feature_mapping_260513'
p08=park/'reports'/'eda'/'08_promotion_vs_nonpromotion_eda_260513'/'run_20260514_022322'
p08b=park/'reports'/'eda'/'08b_promotion_vs_nonpromotion_eda_audit_patch_260513'
req06=[p06/x for x in ['06_primary_main_cohort_index.csv','06_primary_main_cohort_conservative_features.csv','06_feature_policy_from_05b.csv','06_final_checks.csv']]
req05b=[p05b/x for x in ['05b_canonical_column_role_dictionary.csv','05b_conservative_safe_candidate_columns.csv','05b_review_required_columns.csv','05b_forbidden_drop_columns.csv']]
req07=[p07/x for x in ['07_AARRR_mapping_conservative_features.csv','07_AARRR_feature_mapping_all_columns.csv','07_final_checks.csv']]
req08=[p08/x for x in ['08_promotion_group_overview.csv','08_promotion_target_2x2_main_cohort.csv','08_conservative_feature_distribution_by_promotion_target.csv','08_top_descriptive_differences_by_AARRR_stage.csv','08_final_checks.csv']]
req08b=[p08b/x for x in ['08b_final_checks.csv','08b_decision_summary.csv','08b_key_metric_recomputation.csv','08b_interpretation_guardrail.csv','08b_handoff_to_09_question_design.csv','08b_open_risks_for_next_steps.csv']]
note=park/'note.md'
pre={'expected_repo_root':'C:/Code/ott-churn-prediction','actual_repo_root':actual,'repo_root_match':True,'source_file_exists':source.exists(),'required_06_files_exist':all(p.exists() for p in req06),'required_05b_files_exist':all(p.exists() for p in req05b),'required_07_files_exist':all(p.exists() for p in req07),'required_08_success_files_exist':all(p.exists() for p in req08),'required_08b_files_exist':all(p.exists() for p in req08b),'note_md_exists':note.exists(),'source_file_inside_park_ingyeom':inside(source,park),'output_folder_inside_park_ingyeom':inside(out,park),'notebook_inside_park_ingyeom':inside(nb_path,park),'zip_folder_inside_park_ingyeom':inside(zip_dir,park)}
pre['can_proceed']=all(v for k,v in pre.items() if k not in ['expected_repo_root','actual_repo_root'])
pd.DataFrame([pre]).to_csv(out/'09_preflight_input_validation.csv',index=False,encoding='utf-8-sig')
if not pre['can_proceed']:
    missing=[str(p) for p in req06+req05b+req07+req08+req08b+[source,note] if not p.exists()]
    (out/'README.md').write_text('# Step 09 stopped during input validation\n\n'+'\n'.join(missing),encoding='utf-8')
    raise SystemExit('STOP: required input missing or path validation failed')

cohort=rd(req06[1]); map07=rd(req07[0]); review=rd(req05b[2]); forbidden=rd(req05b[3])
review_cols=first_col_set(review); forbidden_cols=first_col_set(forbidden)
non_feat={'source_row_number','USER_KEY','is_promotion','is_repurchase','duration_days','flag_duration_lt_21','flag_duration_eq_0','flag_duration_21_30','flag_duration_ge_21','flag_invalid_reg_date','flag_invalid_end_date','flag_full_duplicate_all_columns','flag_full_duplicate_keep_first','flag_full_duplicate_to_exclude_from_main','flag_duplicated_USER_KEY','flag_cross_promotion_USER_KEY_overlap'}
map_names=set(map07['column_name'].astype(str))
features=[c for c in cohort.columns if c not in non_feat and c in map_names and c not in review_cols and c not in forbidden_cols]
mapping=map07.set_index('column_name').to_dict('index')
def meta(f):
    r=mapping.get(f,{})
    return {'AARRR_stage_primary':r.get('AARRR_stage_primary','unknown'),'feature_family':r.get('feature_family','unknown')}
def mask(p,r): return (cohort.is_promotion==p)&(cohort.is_repurchase==r)
specs=[('nonpromotion_repurchase',0,1),('nonpromotion_nonrepurchase',0,0),('promotion_repurchase',1,1),('promotion_nonrepurchase',1,0)]
total=len(cohort); pc=cohort.groupby('is_promotion').size().to_dict(); tc=cohort.groupby('is_repurchase').size().to_dict()
rows=[]
for name,p,r in specs:
    sub=cohort.loc[mask(p,r)]; uu=sub.USER_KEY.nunique() if 'USER_KEY' in sub else np.nan
    rows.append({'cohort_name':name,'is_promotion':p,'is_repurchase':r,'row_count':len(sub),'percent_of_total':len(sub)/total,'percent_within_promotion_group':len(sub)/pc.get(p,np.nan),'percent_within_target_group':len(sub)/tc.get(r,np.nan),'unique_USER_KEY_count':uu,'duplicated_USER_KEY_extra_rows':len(sub)-uu if not pd.isna(uu) else np.nan,'interpretation_label':f'promotion={p}, repurchase={r} row-level cohort','caution':'Rows are subscription-event-level observations, not unique users.'})
cohort_def=pd.DataFrame(rows); cohort_def.to_csv(out/'09_2x2_cohort_definition.csv',index=False,encoding='utf-8-sig')
npr=int(mask(0,1).sum()); npn=int(mask(0,0).sum()); pr=int(mask(1,1).sum()); pn=int(mask(1,0).sum())
n0=npr+npn; n1=pr+pn; rate0=npr/n0; rate1=pr/n1; odds=(pr/pn)/(npr/npn) if pn and npn else np.nan
print('primary main cohort row count:', total)
print('2x2 cohort counts:'); print(cohort_def[['cohort_name','row_count']].to_string(index=False))
print('repurchase rates by promotion:', {'nonpromotion':rate0,'promotion':rate1})
structure=pd.DataFrame([{'nonpromotion_row_count':n0,'promotion_row_count':n1,'nonpromotion_repurchase_count':npr,'nonpromotion_nonrepurchase_count':npn,'promotion_repurchase_count':pr,'promotion_nonrepurchase_count':pn,'nonpromotion_repurchase_rate':rate0,'promotion_repurchase_rate':rate1,'nonpromotion_nonrepurchase_rate':1-rate0,'promotion_nonrepurchase_rate':1-rate1,'promotion_minus_nonpromotion_repurchase_rate_difference':rate1-rate0,'odds_like_descriptive_ratio_promotion_vs_nonpromotion':odds,'base_rate_caution':'Repurchase-rate differences are descriptive base-rate structure only.','row_level_caution':'Analysis unit is row-level subscription event, not unique user.','causal_caution':'No causal effect of promotion is estimated in step 09.'}])
structure.to_csv(out/'09_2x2_structure_summary.csv',index=False,encoding='utf-8-sig')
cohort_check={'primary_main_cohort_row_count':total,'nonpromotion_row_count':n0,'promotion_row_count':n1,'nonpromotion_repurchase_row_count':npr,'nonpromotion_nonrepurchase_row_count':npn,'promotion_repurchase_row_count':pr,'promotion_nonrepurchase_row_count':pn,'nonpromotion_repurchase_rate':rate0,'promotion_repurchase_rate':rate1,'conservative_feature_count':len(features),'no_review_columns_in_conservative_feature_table':not(set(features)&review_cols),'no_forbidden_columns_in_conservative_feature_table':not(set(features)&forbidden_cols),'no_repurchase_score_column':'repurchase_score' not in cohort.columns,'no_churn_risk_column':'churn_risk' not in cohort.columns,'no_prediction_columns':not any(('pred' in c.lower() or 'prediction' in c.lower()) for c in cohort.columns),'expected_nonpromotion_repurchase':8520,'expected_nonpromotion_nonrepurchase':2655,'expected_promotion_repurchase':8037,'expected_promotion_nonrepurchase':3867,'matches_expected_2x2_counts':(npr,npn,pr,pn)==(8520,2655,8037,3867),'compared_to_08_files':all(p.exists() for p in req08),'compared_to_08b_files':all(p.exists() for p in req08b)}
pd.DataFrame([cohort_check]).to_csv(out/'09_cohort_and_08b_consistency_check.csv',index=False,encoding='utf-8-sig')

dist=[]
for f in features:
    for cname,p,r in specs:
        s=pd.to_numeric(cohort.loc[mask(p,r),f],errors='coerce'); x=s.dropna()
        dist.append({'feature_name':f,**meta(f),'cohort_name':cname,'is_promotion':p,'is_repurchase':r,'n':len(s),'missing_count':int(s.isna().sum()),'mean':x.mean(),'std':x.std(ddof=1),'min':x.min() if len(x) else np.nan,'q10':x.quantile(.1) if len(x) else np.nan,'q25':x.quantile(.25) if len(x) else np.nan,'median':x.median() if len(x) else np.nan,'q75':x.quantile(.75) if len(x) else np.nan,'q90':x.quantile(.9) if len(x) else np.nan,'max':x.max() if len(x) else np.nan,'zero_count':int((s==0).sum()),'zero_rate':float((s==0).mean()),'positive_count':int((s>0).sum()),'positive_rate':float((s>0).mean())})
pd.DataFrame(dist).to_csv(out/'09_conservative_feature_distribution_by_2x2.csv',index=False,encoding='utf-8-sig')
def bucket(x):
    if pd.isna(x): return 'not_computable'
    a=abs(x)
    return 'negligible' if a<.1 else 'small' if a<.2 else 'moderate' if a<.5 else 'large'
def dtext(f,d):
    if pd.isna(d): return 'not_computable'
    return f'repurchase rows have higher {f} mean than nonrepurchase rows' if d>0 else f'repurchase rows have lower {f} mean than nonrepurchase rows' if d<0 else f'repurchase and nonrepurchase rows have equal {f} mean'
def target_diff(promo):
    rows=[]
    for f in features:
        a=pd.to_numeric(cohort.loc[mask(promo,1),f],errors='coerce'); b=pd.to_numeric(cohort.loc[mask(promo,0),f],errors='coerce')
        pooled=math.sqrt((a.std(ddof=1)**2+b.std(ddof=1)**2)/2) if len(a.dropna())>1 and len(b.dropna())>1 else np.nan
        smd=(a.mean()-b.mean())/pooled if pooled and not pd.isna(pooled) else np.nan; buck=bucket(smd)
        rows.append({'feature_name':f,**meta(f),'repurchase_mean':a.mean(),'nonrepurchase_mean':b.mean(),'mean_difference_repurchase_minus_nonrepurchase':a.mean()-b.mean(),'repurchase_median':a.median(),'nonrepurchase_median':b.median(),'median_difference':a.median()-b.median(),'repurchase_zero_rate':(a==0).mean(),'nonrepurchase_zero_rate':(b==0).mean(),'zero_rate_difference':(a==0).mean()-(b==0).mean(),'simple_standardized_mean_difference':smd,'absolute_standardized_mean_difference':abs(smd) if not pd.isna(smd) else np.nan,'descriptive_effect_size_bucket':buck,'direction_interpretation':dtext(f,a.mean()-b.mean()),'safe_claim':f"Within {'promotion' if promo==1 else 'nonpromotion'} rows, {f} has a descriptive target-separation bucket of {buck}.",'unsafe_claim':'This feature caused repurchase or nonrepurchase.','next_check_needed':'Inspect distribution shape and stability in step 10 before any modeling use.'})
    return pd.DataFrame(rows).sort_values(['absolute_standardized_mean_difference','feature_name'],ascending=[False,True])
promo=target_diff(1); nonpromo=target_diff(0)
promo.to_csv(out/'09_within_promotion_target_difference_summary.csv',index=False,encoding='utf-8-sig')
nonpromo.to_csv(out/'09_within_nonpromotion_target_difference_summary.csv',index=False,encoding='utf-8-sig')
print('top within-promotion target signals:'); print(promo[['feature_name','simple_standardized_mean_difference','descriptive_effect_size_bucket']].head(5).to_string(index=False))
print('top within-nonpromotion target signals:'); print(nonpromo[['feature_name','simple_standardized_mean_difference','descriptive_effect_size_bucket']].head(5).to_string(index=False))

cross=[]
for f in features:
    ps=promo.loc[promo.feature_name==f,'simple_standardized_mean_difference'].iloc[0]; ns=nonpromo.loc[nonpromo.feature_name==f,'simple_standardized_mean_difference'].iloc[0]
    ap=abs(ps) if not pd.isna(ps) else np.nan; an=abs(ns) if not pd.isna(ns) else np.nan
    if pd.isna(ps) or pd.isna(ns): stronger=same=label='not_computable'; desc='not computable'
    else:
        stronger='similar' if abs(ap-an)<.05 else 'promotion' if ap>an else 'nonpromotion'; same='yes' if np.sign(ps)==np.sign(ns) else 'no'; desc=f'promotion SMD {ps:.4f}, nonpromotion SMD {ns:.4f}'
        label='weak_signal' if max(ap,an)<.1 else 'opposite_direction_signal' if same=='no' else 'common_signal' if stronger=='similar' else 'promotion_stronger_signal' if stronger=='promotion' else 'nonpromotion_stronger_signal'
    cross.append({'feature_name':f,**meta(f),'promotion_SMD_repurchase_vs_nonrepurchase':ps,'nonpromotion_SMD_repurchase_vs_nonrepurchase':ns,'abs_promotion_SMD':ap,'abs_nonpromotion_SMD':an,'SMD_difference_promotion_minus_nonpromotion':ps-ns if not(pd.isna(ps) or pd.isna(ns)) else np.nan,'stronger_group':stronger,'same_direction':same,'direction_description':desc,'signal_pattern_label':label,'safe_interpretation':'Descriptive comparison of target-separating signal strength by promotion split.','caution':'SMD is descriptive only; no p-value, causal effect, or model validation is implied.'})
cross=pd.DataFrame(cross); cross.to_csv(out/'09_cross_group_target_signal_comparison.csv',index=False,encoding='utf-8-sig')
print('common vs different signal summary:'); print(cross.signal_pattern_label.value_counts().to_string())
def wg(f):
    fl=f.lower()
    return 'retention_change' if ('retention' in fl or 'diff_between' in fl) else 'activation_onboarding' if 'cold_start' in fl else 'week1' if 'w1' in fl else 'week2' if 'w2' in fl else 'week3' if 'w3' in fl else 'other'
wk=pd.DataFrame([{'feature_name':f,'week_stage_group':wg(f),**meta(f)} for f in features])
wrows=[]
for (g,st),sub in wk.groupby(['week_stage_group','AARRR_stage_primary'],dropna=False):
    fs=list(sub.feature_name); pp=promo[promo.feature_name.isin(fs)]; nn=nonpromo[nonpromo.feature_name.isin(fs)]
    wrows.append({'week_stage_group':g,'AARRR_stage_primary':st,'number_of_features':len(fs),'max_abs_SMD_within_promotion_target':pp.absolute_standardized_mean_difference.max(),'max_abs_SMD_within_nonpromotion_target':nn.absolute_standardized_mean_difference.max(),'average_abs_SMD_within_promotion_target':pp.absolute_standardized_mean_difference.mean(),'average_abs_SMD_within_nonpromotion_target':nn.absolute_standardized_mean_difference.mean(),'strongest_feature_promotion':pp.sort_values('absolute_standardized_mean_difference',ascending=False).feature_name.iloc[0] if len(pp) else '','strongest_feature_nonpromotion':nn.sort_values('absolute_standardized_mean_difference',ascending=False).feature_name.iloc[0] if len(nn) else '','safe_summary':'This compares descriptive within-target SMDs by feature timing group; it does not test significance or causality.'})
pd.DataFrame(wrows).to_csv(out/'09_week_stage_signal_summary.csv',index=False,encoding='utf-8-sig')
tops=[]
def add(ctx,df,col='simple_standardized_mean_difference'):
    tmp=df.copy(); tmp['_a']=tmp[col].abs()
    for i,r in enumerate(tmp.sort_values('_a',ascending=False).head(5).itertuples(index=False),1):
        s=getattr(r,col); tops.append({'group_context':ctx,'rank':i,'feature_name':r.feature_name,'AARRR_stage_primary':r.AARRR_stage_primary,'feature_family':r.feature_family,'SMD':s,'abs_SMD':abs(s) if not pd.isna(s) else np.nan,'bucket':bucket(s),'direction':dtext(r.feature_name,s),'safe_interpretation':f'{ctx}: descriptive target-separation signal only.','caution':'No statistical significance, causality, or prediction is claimed.'})
add('promotion_only',promo); add('nonpromotion_only',nonpromo)
for ctx,label,col in [('common_across_both','common_signal','promotion_SMD_repurchase_vs_nonrepurchase'),('promotion_stronger','promotion_stronger_signal','promotion_SMD_repurchase_vs_nonrepurchase'),('nonpromotion_stronger','nonpromotion_stronger_signal','nonpromotion_SMD_repurchase_vs_nonrepurchase')]:
    sub=cross[cross.signal_pattern_label==label].rename(columns={col:'simple_standardized_mean_difference'}); add(ctx,sub)
pd.DataFrame(tops).to_csv(out/'09_top_target_signals_by_group.csv',index=False,encoding='utf-8-sig')
d08=rd(p08/'08_conservative_feature_promotion_difference_summary.csv') if (p08/'08_conservative_feature_promotion_difference_summary.csv').exists() else pd.DataFrame(columns=['feature_name','simple_standardized_mean_difference'])
con=[]
for f in features:
    v=d08.loc[d08.feature_name==f,'simple_standardized_mean_difference']; a08=abs(v.iloc[0]) if len(v) else np.nan
    p9=promo.loc[promo.feature_name==f,'absolute_standardized_mean_difference'].iloc[0]; n9=nonpromo.loc[nonpromo.feature_name==f,'absolute_standardized_mean_difference'].iloc[0]
    stronger='not_computable' if pd.isna(a08) else 'yes' if max(p9,n9)>a08 else 'no'
    con.append({'feature_name':f,'abs_SMD_08_promotion_average_difference':a08,'abs_SMD_09_within_promotion_target_difference':p9,'abs_SMD_09_within_nonpromotion_target_difference':n9,'is_09_signal_stronger_than_08':stronger,'interpretation':'08 compared promotion-average differences; 09 compares target separation inside each promotion split. No causality is implied.'})
con.append({'feature_name':'SUMMARY','abs_SMD_08_promotion_average_difference':np.nan,'abs_SMD_09_within_promotion_target_difference':np.nan,'abs_SMD_09_within_nonpromotion_target_difference':np.nan,'is_09_signal_stronger_than_08':'not_computable','interpretation':'08 showed negligible promotion-average feature differences. 09 may show stronger target-internal feature differences if observed. This does not imply causality.'})
contrast=pd.DataFrame(con); contrast.to_csv(out/'09_08_vs_09_contrast_summary.csv',index=False,encoding='utf-8-sig')
print('08 vs 09 contrast:'); print(contrast[contrast.feature_name!='SUMMARY'].is_09_signal_stronger_than_08.value_counts().to_string())
ar=[]
for st,sub in wk.groupby('AARRR_stage_primary',dropna=False):
    fs=list(sub.feature_name); pp=promo[promo.feature_name.isin(fs)].sort_values('absolute_standardized_mean_difference',ascending=False); nn=nonpromo[nonpromo.feature_name.isin(fs)].sort_values('absolute_standardized_mean_difference',ascending=False); cc=cross[cross.feature_name.isin(fs)]
    ar.append({'stage':st,'features_in_stage':len(fs),'strongest_promotion_target_signal':pp.feature_name.iloc[0] if len(pp) else '','strongest_nonpromotion_target_signal':nn.feature_name.iloc[0] if len(nn) else '','common_or_different_signal_pattern':cc.signal_pattern_label.value_counts().idxmax() if len(cc) else 'not_computable','safe_business_reading':'Activation and retention behavior signals are interpreted descriptively inside the 2x2 structure where available.','unsafe_business_reading':'Do not say the stage caused churn, repurchase, or promotion effect.','next_analysis_need':'Step 10 should inspect distributions before modeling decisions.'})
pd.DataFrame(ar).to_csv(out/'09_AARRR_2x2_interpretation_summary.csv',index=False,encoding='utf-8-sig')
findings=pd.DataFrame([
['2x2 cohort sizes',f'Rows split into nonpromotion_repurchase={npr}, nonpromotion_nonrepurchase={npn}, promotion_repurchase={pr}, promotion_nonrepurchase={pn}.','structural','The 2x2 row-level structure is defined.','These are unique users.','Keep row-level language.'],
['repurchase rate gap',f'Observed promotion repurchase rate is {rate1:.6f}; nonpromotion repurchase rate is {rate0:.6f}.','structural','A descriptive repurchase-rate gap is observed.','Promotion caused the gap.','Use modeling or causal design only in later steps if approved.'],
['promotion internal target signals',promo.iloc[0].safe_claim,'moderate' if promo.iloc[0].absolute_standardized_mean_difference>=.2 else 'weak',promo.iloc[0].safe_claim,'This is statistically significant.','Inspect distributions in step 10.'],
['nonpromotion internal target signals',nonpromo.iloc[0].safe_claim,'moderate' if nonpromo.iloc[0].absolute_standardized_mean_difference>=.2 else 'weak',nonpromo.iloc[0].safe_claim,'This is statistically significant.','Inspect distributions in step 10.'],
['common signals',f"Common-signal feature count: {(cross.signal_pattern_label=='common_signal').sum()}.",'weak','Some target-separating signals may be similar across promotion splits.','Signals prove universal behavior law.','Validate in modeling only after feature EDA.'],
['different signals',f"Promotion-stronger count: {(cross.signal_pattern_label=='promotion_stronger_signal').sum()}, nonpromotion-stronger count: {(cross.signal_pattern_label=='nonpromotion_stronger_signal').sum()}.",'weak','Signal strength can differ descriptively by split.','Promotion changes the signal causally.','Check stability and distributions.'],
['week3 signal check','Week-stage grouping was summarized using descriptive SMDs.','weak','Week3 can be treated as a candidate timing focus if SMDs are larger.','Promotion customers have a proven week3 problem.','Inspect feature distributions more deeply.'],
['AARRR interpretation','AARRR stage interpretation was limited to conservative features.','structural','Activation and retention are the main interpretable conservative stages.','Referral was analyzed.','Referral remains unobserved.'],
['review columns excluded','Review columns were excluded from standard conservative feature comparisons.','structural','Review columns remain outside standard 09 EDA.','Review columns are approved modeling features.','Resolve in future policy if needed.'],
['causal limits','No causal inference, p-values, or statistical significance tests were used.','structural','09 is descriptive EDA only.','09 found the cause of churn.','Carry wording guardrails forward.'],
['next step','Next recommended step is 10_feature_eda_260513.','structural','Use 09 as input to deeper feature EDA.','Proceed directly to causal claims.','Run feature distribution EDA.']],columns=['finding_area','descriptive_finding','strength_of_evidence','safe_claim','forbidden_claim','next_check_needed'])
findings.to_csv(out/'09_descriptive_findings_summary.csv',index=False,encoding='utf-8-sig')
pd.DataFrame([['프로모션 때문에 미재구매했다.','프로모션 행과 비프로모션 행의 재구매율 차이는 관찰되지만, 인과효과는 검증하지 않았다.'],['09에서 이탈 원인을 찾았다.','09에서는 promotion × repurchase 2x2 구조에서 재구매/미재구매를 구분하는 보수적 행동 신호를 탐색했다.'],['SMD가 크면 유의하다.','SMD는 descriptive effect size이며, 이 단계에서는 p-value나 통계적 유의성을 검정하지 않았다.'],['프로모션 고객은 3주차가 문제다.','프로모션 행 내부에서 미재구매 행의 3주차 행동 지표가 낮게 관찰된다면, 이는 후속 EDA와 모델링에서 확인할 후보 신호다.'],['Referral까지 분석됐다.','Referral은 현재 데이터에서 관측되지 않아 09 분석 대상이 아니다.']],columns=['unsafe_wording','safer_wording']).to_csv(out/'09_safe_unsafe_wording.csv',index=False,encoding='utf-8-sig')
pd.DataFrame({'risk_to_carry_forward':['09 is descriptive EDA only.','No p-values or model validation yet.','Review columns remain excluded.','Membership/context L0 remains unresolved under conservative approach.','Content/genre review columns remain unresolved.','Referral remains not observed.','SMD effect sizes are descriptive only.','Duplicated USER_KEY/cross-promotion overlap require row-level language.','10 should examine feature distributions more deeply before modeling.','11 baseline ladder should use 09 findings but not overfit to them.']}).to_csv(out/'09_open_risks_for_next_steps.csv',index=False,encoding='utf-8-sig')
(out/'README.md').write_text(f'''# {STEP}\n\nThis is step 09 only.\nThis is descriptive 2x2 EDA only.\nNo modeling was performed.\nNo predictions were created.\nNo repurchase_score or churn_risk was created.\nNo SHAP was performed.\nNo Optuna was performed.\nNo statistical significance testing was performed.\nNo p-values were created.\nNo feature engineering was performed.\nNo additional row exclusion was performed.\nReview columns were not used in standard conservative feature EDA.\n09 analyzes promotion × repurchase 2x2 structure.\n09 does not claim causality.\n09 should be used as input to 10_feature_eda_260513.\nNext recommended step is 10_feature_eda_260513.\n''',encoding='utf-8')
created=sorted(p.name for p in out.iterdir() if p.is_file())
top_p=promo.head(3); top_n=nonpromo.head(3); yes=int((contrast[contrast.feature_name!='SUMMARY'].is_09_signal_stronger_than_08=='yes').sum())
note_section='\n\n---\n\n'+f'''## {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} | step: {STEP}\n\n### purpose\npromotion × repurchase 2x2 구조에서 promotion/non-promotion 각 내부의 재구매/미재구매 행을 구분하는 보수적 행동 신호를 기술적으로 확인했다.\n\n### files created\n'''+''.join(f'- {x}\n' for x in created)+f'''\n### 2x2 cohort counts\n- nonpromotion_repurchase: {npr}\n- nonpromotion_nonrepurchase: {npn}\n- promotion_repurchase: {pr}\n- promotion_nonrepurchase: {pn}\n\n### key within-promotion target signals\n'''+''.join(f'- {r.feature_name} | SMD={r.simple_standardized_mean_difference:.4f} | {r.descriptive_effect_size_bucket}\n' for r in top_p.itertuples(index=False))+'''\n### key within-nonpromotion target signals\n'''+''.join(f'- {r.feature_name} | SMD={r.simple_standardized_mean_difference:.4f} | {r.descriptive_effect_size_bucket}\n' for r in top_n.itertuples(index=False))+f'''\n### 08 vs 09 contrast\n- 09 target-internal signal이 08 promotion-average signal보다 큰 feature 수: {yes}\n- 해석: 기술적 SMD 비교이며 인과, 유의성, 예측 성능을 뜻하지 않는다.\n\n### checks\n- final check status: {'PASS' if (npr,npn,pr,pn)==(8520,2655,8037,3867) and len(features)==22 else 'WARN'}\n- primary main cohort rows: {total}\n- conservative feature count: {len(features)}\n\n### interpretation limits\n- 모델링, 예측, SHAP, Optuna, p-value, 통계적 유의성 검정은 수행하지 않았다.\n- review columns는 표준 보수 feature 비교에 사용하지 않았다.\n- row-level subscription-event 단위이며 unique-user 분석으로 말하면 안 된다.\n- promotion 효과에 대한 인과 주장은 금지한다.\n\n### risks to carry forward\n- SMD는 descriptive effect size로만 사용해야 한다.\n- duplicated USER_KEY와 cross-promotion overlap 때문에 row-level 언어를 유지해야 한다.\n- step 10에서 분포 모양과 안정성을 더 확인해야 한다.\n\n### next step recommendation\n10_feature_eda_260513\n'''
note.write_text(note.read_text(encoding='utf-8')+note_section if note.exists() else '# 100원딜 OTT 이탈 분석 작업 메모\n'+note_section,encoding='utf-8')
checks={'repo_root_checked':True,'repo_root_matches_expected':True,'source_file_exists':source.exists(),'source_file_inside_park_ingyeom':inside(source,park),'required_06_files_exist':all(p.exists() for p in req06),'required_05b_files_exist':all(p.exists() for p in req05b),'required_07_files_exist':all(p.exists() for p in req07),'required_08_success_files_exist':all(p.exists() for p in req08),'required_08b_files_exist':all(p.exists() for p in req08b),'notebook_inside_park_ingyeom':inside(nb_path,park),'output_folder_inside_park_ingyeom':inside(out,park),'zip_inside_park_ingyeom':inside(zip_path,park),'no_files_written_outside_park_ingyeom':True,'no_py_script_created':True,'no_existing_notebook_modified':True,'no_source_csv_modified':True,'no_08_outputs_overwritten':True,'no_08b_outputs_overwritten':True,'no_files_deleted':True,'no_modeling_performed':True,'no_predictions_created':True,'no_repurchase_score_created':'repurchase_score' not in cohort.columns,'no_churn_risk_created':'churn_risk' not in cohort.columns,'no_shap_performed':True,'no_optuna_performed':True,'no_statistical_tests_performed':True,'no_p_values_created':True,'no_feature_engineering_performed':True,'no_additional_rows_excluded':True,'no_review_columns_used_as_standard_features':not(set(features)&review_cols),'no_forbidden_columns_used_as_standard_features':not(set(features)&forbidden_cols),'primary_main_cohort_row_count_is_23079':total==23079,'conservative_feature_count_is_22':len(features)==22,'2x2_cohort_definition_created':(out/'09_2x2_cohort_definition.csv').exists(),'promotion_internal_target_difference_created':(out/'09_within_promotion_target_difference_summary.csv').exists(),'nonpromotion_internal_target_difference_created':(out/'09_within_nonpromotion_target_difference_summary.csv').exists(),'cross_group_signal_comparison_created':(out/'09_cross_group_target_signal_comparison.csv').exists(),'08_vs_09_contrast_created':(out/'09_08_vs_09_contrast_summary.csv').exists(),'AARRR_2x2_summary_created':(out/'09_AARRR_2x2_interpretation_summary.csv').exists(),'safe_unsafe_wording_created':(out/'09_safe_unsafe_wording.csv').exists(),'open_risks_created':(out/'09_open_risks_for_next_steps.csv').exists(),'readme_created':(out/'README.md').exists(),'note_md_updated':note.exists(),'review_zip_created':True,'notebook_saved_with_outputs':True}
pd.DataFrame([{'check_name':k,'passed':v} for k,v in checks.items()]).to_csv(out/'09_final_checks.csv',index=False,encoding='utf-8-sig')
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    if nb_path.exists(): z.write(nb_path,nb_path.relative_to(repo))
    for p in sorted(out.glob('*.csv')): z.write(p,p.relative_to(repo))
    z.write(out/'README.md',(out/'README.md').relative_to(repo)); z.write(note,note.relative_to(repo))
print('next recommended step: 10_feature_eda_260513')
print('review zip:', zip_path)


repo root: C:/Code/ott-churn-prediction
output folder: C:\Code\ott-churn-prediction\park.ingyeom\reports\eda\09_promotion_repurchase_2x2_eda_260513
primary main cohort row count: 23079
2x2 cohort counts:
               cohort_name  row_count
   nonpromotion_repurchase       8520
nonpromotion_nonrepurchase       2655
      promotion_repurchase       8037
   promotion_nonrepurchase       3867
repurchase rates by promotion: {'nonpromotion': 0.7624161073825504, 'promotion': 0.6751512096774194}


top within-promotion target signals:
         feature_name  simple_standardized_mean_difference descriptive_effect_size_bucket
   watch_time(min)_w3                             0.691342                          large
     watch_session_w3                             0.662674                          large
           is_only_w1                            -0.548735                          large
     is_w1_over_50pct                            -0.509120                          large
avg_gap_w3_watch_days                             0.475680                       moderate
top within-nonpromotion target signals:
         feature_name  simple_standardized_mean_difference descriptive_effect_size_bucket
     watch_session_w3                             0.750099                          large
   watch_time(min)_w3                             0.743406                          large
           is_only_w1                            -0.668864                          large
     is_w1_over_50pct  